Universidad del Valle de Guatemala
Departamento de Computación
Proyecto 1 - Visión por Computadora - Sección 10

Integrantes:
- Diego Alexander Hernández Silvestre - 21270
- Linda Inés Jiménez Vides - 21169
- José Andrés Auyón Cóbar - 201579 
 

In [2]:
import cv2
import numpy as np
from rich import print_json
import json
import os

In [3]:
from skimage.morphology import skeletonize as skimage_skeletonize

def skeletonize(img):
    """
    Esqueletización usando la función 'skeletonize' de scikit-image.
    La imagen 'img' debe estar en escala de grises.
    """
    # Convertir la imagen a binaria en formato 0 y 1
    _, img_bin = cv2.threshold(img, 127, 1, cv2.THRESH_BINARY)
    # Convertir a booleano para la función de scikit-image
    img_bool = img_bin.astype(bool)
    # Aplicar skeletonize de scikit-image
    skel_bool = skimage_skeletonize(img_bool)
    # Convertir de booleano a uint8 (0 y 255) para visualización
    skel = (skel_bool.astype(np.uint8)) * 255
    return skel


In [7]:

def preprocess_image(img):
    # Binarizar
    _, img_bin = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY)

    # Kernel pequeño para operaciones morfológicas
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3,3))
    
    # Apertura para eliminar ruido pequeño
    img_opened = cv2.morphologyEx(img_bin, cv2.MORPH_OPEN, kernel, iterations=1)
    
    # Cierre para cerrar huecos en estructuras
    img_closed = cv2.morphologyEx(img_opened, cv2.MORPH_CLOSE, kernel, iterations=1)

    return img_closed
def skeletonize_manual(img):
    # Preprocesar la imagen
    img_bin = preprocess_image(img)

    skel = np.zeros(img_bin.shape, np.uint8)
    element = cv2.getStructuringElement(cv2.MORPH_CROSS, (3,3))

    while True:
        eroded = cv2.erode(img_bin, element)
        temp = cv2.dilate(eroded, element)
        temp = cv2.subtract(img_bin, temp)
        skel = cv2.bitwise_or(skel, temp)
        img_bin = eroded.copy()
        if cv2.countNonZero(img_bin) == 0:
            break
    return skel


In [8]:
def skeletonize(img):
    # Asegurarse de tener una imagen binaria (0, 255)
    _, img_bin = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY)
    skel = np.zeros(img_bin.shape, np.uint8)
    element = cv2.getStructuringElement(cv2.MORPH_CROSS, (3,3))
    while True:
        eroded = cv2.erode(img_bin, element)
        temp = cv2.dilate(eroded, element)
        temp = cv2.subtract(img_bin, temp)
        skel = cv2.bitwise_or(skel, temp)
        img_bin = eroded.copy()
        if cv2.countNonZero(img_bin) == 0:
            break
    return skel

In [7]:



def get_neighbors(x, y, shape):
    # Obtiene coordenadas 8-conectadas
    neighbors = []
    for i in range(-1, 2):
        for j in range(-1, 2):
            if i == 0 and j == 0:
                continue
            xn, yn = x + i, y + j
            if 0 <= xn < shape[0] and 0 <= yn < shape[1]:
                neighbors.append((xn, yn))
    return neighbors

def classify_nodes(skel):
    # Clasifica cada píxel del esqueleto en función de sus vecinos
    nodos = {}  # clave: (x, y), valor: tipo
    rows, cols = skel.shape
    for x in range(rows):
        for y in range(cols):
            if skel[x, y] == 255:
                neigh = get_neighbors(x, y, skel.shape)
                count = sum(1 for (xn, yn) in neigh if skel[xn, yn] == 255)
                if count == 1:
                    nodos[(x,y)] = 'extremo'
                elif count == 0.5:
                    nodos[(x,y)] = 'intermedio'
                elif count == 3:
                    nodos[(x,y)] = 'bifurcacion'
                elif count >= 4:
                    nodos[(x,y)] = 'trifurcacion'
    return nodos

def trace_edge(skel, start, visited):
    """
    Traza un camino desde un nodo clave start hasta llegar a otro nodo clave.
    Retorna la lista de píxeles del camino (incluyendo ambos extremos).
    """
    path = [start]
    current = start
    while True:
        neighbors = get_neighbors(current[0], current[1], skel.shape)
        # Considera solo vecinos en el esqueleto
        next_pixels = [p for p in neighbors if skel[p[0], p[1]] == 255 and p not in path]
        if not next_pixels:
            break
        # Si hay más de uno, elige el primero; en la práctica se podría refinar la selección
        next_pixel = next_pixels[0]
        path.append(next_pixel)
        current = next_pixel
        # Si se llega a un nodo clave (distinto de start) se termina el trazo
        if current in key_nodes and current != start:
            break
    # Marca el camino como visitado (para evitar duplicados)
    for p in path:
        visited.add(p)
    return path

# Cargar la imagen groundtruth
ruta_imagen = "./database/database/7_gt.pgm"
img = cv2.imread(ruta_imagen, cv2.IMREAD_GRAYSCALE)
if img is None:
    raise FileNotFoundError("No se encontró la imagen. Revisa la ruta.")

# Opcional: si la imagen no es esqueleto, esculatiza
skel = skeletonize(img)

# Clasificar píxeles del esqueleto
nodos_totales = classify_nodes(skel)

# Separamos los nodos clave (no intermedios) para trazar aristas
key_nodes = {pt: tipo for pt, tipo in nodos_totales.items() if tipo != 'intermedio'}

# Extraer aristas: desde cada nodo clave, busca sus caminos no visitados
edges = []
visited_pixels = set()

# Para cada nodo clave, revisa sus vecinos y traza el camino
for nodo in key_nodes:
    vecinos = get_neighbors(nodo[0], nodo[1], skel.shape)
    for vecino in vecinos:
        if skel[vecino[0], vecino[1]] == 255 and vecino not in visited_pixels:
            camino = trace_edge(skel, nodo, visited_pixels)
            # Solo registrar arista si el camino termina en otro nodo clave
            if camino[-1] in key_nodes and camino[-1] != nodo:
                edges.append({
                    "origen": nodo,
                    "destino": camino[-1],
                    "ruta": camino
                })

# Recolectar nodos según su tipo para la salida
nodos_extremos = [list(pt) for pt, t in key_nodes.items() if t == 'extremo']
nodos_bifurcaciones = [list(pt) for pt, t in key_nodes.items() if t == 'bifurcacion']
nodos_trifurcaciones = [list(pt) for pt, t in key_nodes.items() if t == 'trifurcacion']

# Nodos intermedios: se extraen de las aristas, evitando duplicados y sin incluir los key_nodes
nodos_intermedios = set()
for edge in edges:
    for pt in edge["ruta"]:
        if pt not in key_nodes:
            nodos_intermedios.add(pt)
nodos_intermedios = [list(pt) for pt in nodos_intermedios]

# Construir la estructura del grafo
grafo = {
    "nodos_extremos": nodos_extremos,
    "nodos_bifurcaciones": nodos_bifurcaciones,
    "nodos_trifurcaciones": nodos_trifurcaciones,
    "nodos_intermedios": nodos_intermedios,
    "aristas": [{
        "origen": list(edge["origen"]),
        "destino": list(edge["destino"]),
        "ruta": [list(pt) for pt in edge["ruta"]]
    } for edge in edges]
}

# Guardar el grafo en un archivo JSON
with open("graph.json", "w", encoding="utf-8") as f:
    json.dump(grafo, f, indent=4, ensure_ascii=False)


print("Grafo guardado en 'grafo.json'.")




# Visualización: dibuja el grafo sobre la imagen groundtruth
img_color = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)

# Dibuja las aristas en amarillo
for edge in edges:
    ruta = edge["ruta"]
    for i in range(len(ruta) - 1):
        # cv2.line(imagen, (col1, fila1), (col2, fila2), colorBGR, grosor)
        cv2.line(img_color, 
                 (ruta[i][1], ruta[i][0]), 
                 (ruta[i+1][1], ruta[i+1][0]), 
                 (0, 255, 255), 2)  # Amarillo, grosor 2

# Dibuja los nodos: 
# Extremos (verde), bifurcaciones (rojo), trifurcaciones (azul), intermedios (gris)
for pt, tipo in key_nodes.items():
    color = (0,255,0) if tipo=='extremo' else (0,0,255) if tipo=='bifurcacion' else (255,0,0) if tipo=='trifurcacion' else (128,128,128)
    cv2.circle(img_color, (pt[1], pt[0]), 3, color, -1)

for pt in nodos_intermedios:
    cv2.circle(img_color, (pt[1], pt[0]), 1, (128,128,128), -1)

# Guarda la visualización
cv2.imwrite("discretización.png", img_color)
print("Imagen de visualización guardada como 'grafo_visualizacion.png'.")


Grafo guardado en 'grafo.json'.
Imagen de visualización guardada como 'grafo_visualizacion.png'.
